# PaperHub Jupyter Başlangıç Defteri (Türkçe)

Bu defterin amacı PaperHub paketini Türkçe çıktı seçeneğiyle sıfırdan çalıştırmak için adım adım yol göstermektir.

Akış:

1. Notebook'un repo kökünde çalıştığından emin olacağız.
2. Paketi editable modda kuracağız.
3. API anahtarını güvenli biçimde vereceğiz.
4. Kurulum, import, launcher ve test kontrollerini çalıştıracağız.
5. HuggingFace Daily Papers metadata akışını LLM çağırmadan kontrol edeceğiz.
6. API anahtarı hazırsa gerçek Türkçe özetleme çalıştıracağız.
7. `PaperHub.run`, tarih verme biçimleri, çıktı formatlama, cache ve async kullanımını göreceğiz.

Not: API anahtarını notebook içine düz metin olarak yazma. Aşağıdaki `getpass` hücresi anahtarı gizli girdi olarak alır ve sadece bu kernel oturumu boyunca environment variable olarak tutar.


## 1. Repo Kökünü Bul

Bu hücre notebook hangi klasörden açılmış olursa olsun `pyproject.toml` ve `src/paperhub` içeren proje kökünü bulur ve çalışma dizinini oraya taşır.


In [ ]:
import os
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paperhub").exists():
            return candidate
    raise RuntimeError("PaperHub repo kökü bulunamadı. Notebook'u proje içinde açtığından emin ol.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
os.chdir(REPO_ROOT)
print(f"Çalışma dizini: {REPO_ROOT}")

## 2. Paketi Kur

Bu hücre paketi editable modda kurar. Böylece `src/paperhub` içindeki değişiklikler tekrar kurulum yapmadan notebook'a yansır.

Varsayılan provider `openai`, varsayılan model `gpt-5.4-mini` ve reasoning effort `xhigh` olarak ayarlanmıştır. Anthropic veya Google kullanacaksan aşağıdaki `PROVIDER` değerini değiştir; model belirtmezsen PaperHub seçilen provider'ın kendi default modelini kullanır.


In [ ]:
import subprocess
import sys

# Seçenekler: "openai", "anthropic", "google"
PROVIDER = "openai"

# Çıktı dili: İngilizce için "en", Türkçe için "tr".
LANGUAGE = "tr"

# Model belirtmek zorunda değilsin. None bırakırsan seçilen provider'ın default modeli kullanılır.
# Örnek override: MODEL = "gpt-5.4-mini"
MODEL = None

DEFAULT_MODEL_BY_PROVIDER = {
    "anthropic": "claude-haiku-4-5-20251001",
    "openai": "gpt-5.4-mini",
    "google": "gemini-3-flash-preview",
}

subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[dev]"])

if PROVIDER == "anthropic":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[anthropic]"])
elif PROVIDER == "google":
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".[google]"])

print(f"Kurulum tamam: provider={PROVIDER}, dil={LANGUAGE}, model_override={MODEL or 'yok'}")

## 3. API Anahtarını Ver

İki güvenli yol var.

### Yol A: Notebook oturumu için gizli giriş

Aşağıdaki hücre önce proje kökündeki `.env` dosyasını kontrol eder. Seçili provider için API anahtarı `.env` içinde varsa sana anahtar sormaz; anahtarı bu kernel oturumu için environment variable'a taşır.

`.env` içinde anahtar yoksa `getpass` ile anahtarı gizli alır. Anahtar notebook dosyasına kaydolmaz; sadece açık kernel oturumunda environment variable olur.

### Yol B: `.env` dosyası

Terminalde veya editörde proje kökünde `.env.example` dosyasını `.env` olarak kopyalayabilirsin:

```bash
cp .env.example .env
```

Sonra ilgili satırı doldur:

- OpenAI: `OPENAI_API_KEY=...`
- Anthropic: `ANTHROPIC_API_KEY=...`
- Google: `GOOGLE_API_KEY=...`

`.env` dosyası git'e eklenmemeli; repo `.gitignore` içinde bunu zaten dışarıda bırakıyor.


In [ ]:
import os
from getpass import getpass

from dotenv import dotenv_values

API_KEY_ENV_BY_PROVIDER = {
    "anthropic": "ANTHROPIC_API_KEY",
    "openai": "OPENAI_API_KEY",
    "google": "GOOGLE_API_KEY",
}

api_key_env = API_KEY_ENV_BY_PROVIDER[PROVIDER]
env_path = REPO_ROOT / ".env"
env_values = dotenv_values(env_path) if env_path.exists() else {}
key_source = "environment"

env_file_key = (env_values.get(api_key_env) or "").strip()
if not os.environ.get(api_key_env) and env_file_key:
    os.environ[api_key_env] = env_file_key
    key_source = ".env"
elif not os.environ.get(api_key_env):
    value = getpass(f"{api_key_env} gir (gizli): ").strip()
    if value:
        os.environ[api_key_env] = value
        key_source = "getpass"
    else:
        key_source = "eksik"

MODEL_ENV_BY_PROVIDER = {
    "anthropic": "PAPERHUB_ANTHROPIC_MODEL",
    "openai": "PAPERHUB_OPENAI_MODEL",
    "google": "PAPERHUB_GOOGLE_MODEL",
}

provider_model_env = MODEL_ENV_BY_PROVIDER[PROVIDER]
env_file_provider_model = (env_values.get(provider_model_env) or "").strip()
env_file_global_model = (env_values.get("PAPERHUB_MODEL") or "").strip()

if MODEL is None and env_file_provider_model:
    os.environ[provider_model_env] = env_file_provider_model
elif MODEL is None:
    os.environ.setdefault(provider_model_env, DEFAULT_MODEL_BY_PROVIDER[PROVIDER])

os.environ["PAPERHUB_PROVIDER"] = PROVIDER
if MODEL is None:
    os.environ.pop("PAPERHUB_MODEL", None)
else:
    os.environ["PAPERHUB_MODEL"] = MODEL
os.environ.setdefault("PAPERHUB_CONCURRENCY", "2")
os.environ.setdefault("PAPERHUB_MAX_PDF_CHARS", "60000")

effective_model_hint = (
    MODEL or os.environ.get(provider_model_env) or DEFAULT_MODEL_BY_PROVIDER[PROVIDER]
)
if MODEL is None and env_file_global_model:
    print(
        "Not: PAPERHUB_MODEL .env içinde dolu. Model provider ile uyumluysa PaperHub bunu kullanır; "
        "uyumsuzsa seçili provider'ın default modeli kullanılır."
    )

print(f"Provider: {PROVIDER}")
print(f"Çıktı dili: {LANGUAGE}")
print(f"Model override: {MODEL or 'yok'}")
print(f"Provider default model adayı: {effective_model_hint}")
print(f"{api_key_env}: {'hazır' if os.environ.get(api_key_env) else 'eksik'} ({key_source})")

## 4. Hazırlık Kontrolü

Bu bölüm paketin import edilebildiğini, ayarların okunabildiğini ve hızlı unit testlerin geçtiğini kontrol eder. Bu kontroller gerçek LLM çağrısı yapmaz.


In [ ]:
import httpx

import paperhub
from paperhub import PaperHub, RunRequest, load_settings, render_markdown, render_plain
from paperhub.dates import resolve_range
from paperhub.fetchers import HFAPIFetcher, HFHtmlFetcher, papers_in_range

settings = load_settings()

print("paperhub version:", paperhub.__version__)
print("package path:", paperhub.__file__)
print("settings provider:", settings.paperhub_provider)
print("settings global model override:", settings.paperhub_model or "yok")
print("effective provider model:", settings.model_for_provider(PROVIDER))
print("cache dir:", settings.cache_dir())

request = RunRequest(period="month", year=2026, month=5, top_n=3, language=LANGUAGE)
print("programatik request örneği:", request)

assert hasattr(paperhub, "PaperHub")
assert request.period == "month"
print("Import ve programatik request kontrolü geçti.")

In [ ]:
quick_tests = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_public_api.py",
        "tests/test_nl_parser.py",
        "tests/test_dates.py",
    ],
    text=True,
    capture_output=True,
)

print(quick_tests.stdout)
if quick_tests.stderr:
    print(quick_tests.stderr)
assert quick_tests.returncode == 0, "Hızlı testlerden biri başarısız oldu. Üstteki çıktıyı incele."
print("Hızlı test kontrolü geçti.")

## 5. HuggingFace Metadata Kontrolü

Bu smoke test sadece HuggingFace Daily Papers metadata çeker. PDF indirme veya LLM çağrısı yapmaz.

Amaç: internet bağlantısı ve HuggingFace fetch pipeline hazır mı görmek.

Hücre birkaç aday tarihi programatik metadata yolu ile dener, sonra bulunan tarih için programatik argümanları `LIVE_RUN_KWARGS` olarak saklar. Sonraki gerçek özetleme hücresi bu argümanları kullanır.


In [ ]:
CANDIDATE_RUNS = [
    {
        "label": "2024-05-01",
        "kwargs": {"period": "day", "year": 2024, "month": 5, "day": 1, "top_n": 1},
    },
    {
        "label": "2024-10-01",
        "kwargs": {"period": "day", "year": 2024, "month": 10, "day": 1, "top_n": 1},
    },
    {
        "label": "Mayıs 2026",
        "kwargs": {"period": "month", "year": 2026, "month": 5, "top_n": 1},
    },
]

selected_run = None

for candidate in CANDIDATE_RUNS:
    request = RunRequest(**candidate["kwargs"], language=LANGUAGE)
    start_date, end_date = resolve_range(request)
    print(f"\n--- Deneniyor: {candidate['label']} ({start_date} - {end_date})")
    async with httpx.AsyncClient(
        follow_redirects=True, timeout=settings.paperhub_request_timeout_s
    ) as client:
        primary = HFAPIFetcher(client)
        fallback = HFHtmlFetcher(client)
        papers = await papers_in_range(
            primary, start_date, end_date, request.top_n, fallback=fallback
        )
    print(f"{len(papers)} paper bulundu")
    if papers:
        selected_run = candidate
        break

if selected_run is None:
    raise RuntimeError(
        "Metadata smoke test paper bulamadı. İnternet bağlantısını kontrol et veya CANDIDATE_RUNS listesindeki tarihleri değiştir."
    )

LIVE_RUN_KWARGS = selected_run["kwargs"]
LIVE_REQUEST = RunRequest(**LIVE_RUN_KWARGS, language=LANGUAGE)
print(
    f"\nGerçek LLM çalıştırması için seçilen programatik çalışma: {selected_run['label']} -> {LIVE_RUN_KWARGS}"
)

## 6. PaperHub Nesnesini Başlat

`hub = PaperHub(...)` ana giriş noktasıdır.

Önemli parametreler:

- `provider`: `openai`, `anthropic`, veya `google`.
- `model`: sağlayıcıdaki model id.
- `language`: Türkçe çıktı için `"tr"`, İngilizce çıktı için `"en"`.
- `concurrency`: aynı anda kaç paper ajanı çalışsın. İlk deneme için `1` veya `2` daha kontrollüdür.
- `max_pdf_chars`: PDF metninden modele gönderilecek üst karakter sınırı.
- `cache_dir`: cache klasörünü özel vermek istersen kullanılır.

Aynı paper, model ve dil ikinci kez çalıştırılırsa özet cache'den gelir; tekrar LLM ücreti doğurmaz.


In [ ]:
hub = PaperHub(
    provider=PROVIDER,
    model=MODEL,
    language=LANGUAGE,
    concurrency=2,
    max_pdf_chars=60000,
)

print("PaperHub hazır.")
print("Provider:", hub.provider)
print("Model:", hub.model)
print("Dil:", hub.language)
print("Concurrency:", hub.concurrency)
print("Cache root:", hub.cache.root)

## 7. Gerçek Çalıştırma: `hub.run(...)`

Bu hücre gerçek pipeline'ı çalıştırır:

1. Tarih sorgusunu parse eder.
2. HuggingFace Daily Papers metadata çeker.
3. İlgili paper PDF'ini arXiv'den indirir.
4. PDF metnini çıkarır.
5. Her paper için ayrı `PaperAgent` çalıştırır.
6. LLM'den Türkçe, yapılandırılmış özet alır.
7. Notebook içinde Markdown olarak gösterir.
8. Aynı zamanda `list[PaperSummary]` döndürür.

`top 1` ile başlamak maliyeti ve bekleme süresini düşük tutar. Daha sonra `top 5`, `top 10` gibi artırabilirsin.


In [ ]:
assert os.environ.get(api_key_env), (
    f"{api_key_env} eksik. 3. adımdaki API anahtarı hücresini çalıştır."
)

summaries = hub.run(**LIVE_RUN_KWARGS, display=True, language=LANGUAGE)

print(f"Dönen özet sayısı: {len(summaries)}")
for idx, summary in enumerate(summaries, start=1):
    status = "HATA" if summary.error else "OK"
    print(f"{idx}. {status} | {summary.arxiv_id} | {summary.title}")
    if summary.error:
        print("   error:", summary.error)

## 8. Text Çıktı Almak

`hub.run(..., display=True)` Jupyter'da Markdown render eder. Düz metin istiyorsan iki yol var:

1. `hub.run(..., display=False)` ile sadece listeyi al.
2. `render_plain(summaries, request)` ile terminal dostu text üret.

`render_markdown` ise Markdown string üretir; dosyaya yazmak veya başka yerde göstermek için kullanılır.


In [ ]:
request = LIVE_REQUEST

plain_text = render_plain(summaries, request)
markdown_text = render_markdown(summaries, request)

print("--- Plain text çıktı ---")
print(plain_text[:4000])

print("\nMarkdown karakter sayısı:", len(markdown_text))

## 9. Dönen Nesneleri İncele

`hub.run(...)` çıktısı `list[PaperSummary]` tipindedir. Her elemanda şu alanlar bulunur:

- `arxiv_id`
- `title`
- `motivation`
- `method`
- `findings`
- `real_world_examples`
- `summary` - en fazla 6000 karakterlik Türkçe özet
- `language`
- `pdf_chars`
- `model_used`
- `elapsed_s`
- `error` - o paper'da hata varsa dolu olur; diğer paper'lar yine devam eder


In [ ]:
if summaries:
    first = summaries[0]
    data = first.model_dump()
    for key, value in data.items():
        if isinstance(value, str) and len(value) > 500:
            value = value[:500] + "..."
        print(f"{key}: {value}")
else:
    print("Özet listesi boş. LIVE_RUN_KWARGS değerini değiştirip tekrar dene.")

## 10. Tarih Verme Biçimleri

Python kodunda programatik tarih argümanlarını kullan:

```python
hub.run(period="day", year=2026, month=5, day=1, top_n=5)
hub.run(period="week", year=2026, week=18, top_n=5)
hub.run(period="month", year=2026, month=5, top_n=10)
hub.run(period="year", year=2026, top_n=20)
hub.run(period="custom", start=date(2026, 4, 15), end=date(2026, 4, 30), top_n=15)
```

Doğal dil tarih ifadeleri interactive launcher tarafında kalır.


In [ ]:
example_requests = [
    {"period": "day", "year": 2026, "month": 5, "day": 1, "top_n": 5},
    {"period": "week", "year": 2026, "week": 18, "top_n": 5},
    {"period": "month", "year": 2026, "month": 5, "top_n": 10},
    {"period": "year", "year": 2026, "top_n": 20},
]

for kwargs in example_requests:
    print(kwargs, "->", RunRequest(**kwargs, language=LANGUAGE))

## 11. Ek Canlı Örnekler

Aşağıdaki hücre varsayılan olarak kapalıdır, çünkü her açık örnek yeni paper/model/dil kombinasyonlarında LLM çağrısı yapabilir.

`RUN_MORE_LIVE_EXAMPLES = True` yaparsan programatik tarih biçimlerini gerçekten çalıştırır.


In [ ]:
from datetime import date

RUN_MORE_LIVE_EXAMPLES = False

if RUN_MORE_LIVE_EXAMPLES:
    day_summaries = hub.run(period="day", year=2024, month=5, day=1, top_n=1, display=False)
    week_summaries = hub.run(period="week", year=2024, week=18, top_n=1, display=False)
    month_summaries = hub.run(period="month", year=2024, month=5, top_n=1, display=False)
    custom_summaries = hub.run(
        period="custom",
        start=date(2024, 5, 1),
        end=date(2024, 5, 3),
        top_n=1,
        display=False,
    )
    print(len(day_summaries), len(week_summaries), len(month_summaries), len(custom_summaries))
else:
    print("Ek canlı örnekler kapalı. Çalıştırmak için RUN_MORE_LIVE_EXAMPLES = True yap.")

## 12. Async Kullanım: `await hub.arun(...)`

Jupyter zaten async desteklediği için istersen doğrudan `await hub.arun(...)` kullanabilirsin.

`hub.run(...)` normal notebook kullanımında da çalışır; paket Jupyter event loop çakışmasını kendi içinde yönetir.

Aşağıdaki hücre varsayılan olarak kapalıdır; açarsan tekrar canlı çalışma yapabilir.


In [ ]:
RUN_ASYNC_EXAMPLE = False

if RUN_ASYNC_EXAMPLE:
    async_summaries = await hub.arun(**LIVE_RUN_KWARGS, language=LANGUAGE)
    print(f"Async özet sayısı: {len(async_summaries)}")
else:
    print("Async örnek kapalı. Çalıştırmak için RUN_ASYNC_EXAMPLE = True yap.")

## 13. Cache Kontrolü

PaperHub şunları cache'ler:

- Paper metadata
- PDF metni
- Model ve dil bazlı özetler

Aynı `arxiv_id`, aynı `model` ve aynı `language` ile ikinci çalışma, mümkünse cache'den döner. Modeli veya çıktı dilini değiştirirsen özet yeniden üretilir.


In [ ]:
print("Cache root:", hub.cache.root)
print("SQLite DB:", hub.cache.db_path)
print("PDF cache:", hub.cache.pdf_dir)

if summaries:
    cached = hub.cache.get_summary(summaries[0].arxiv_id, hub.model, LANGUAGE)
    print("İlk özet cache'de mi?:", cached is not None)

## 14. Sağlayıcı Değiştirme

Başka sağlayıcı kullanmak için baştaki `PROVIDER` ve `MODEL` değerlerini değiştirip notebook'u yukarıdan aşağı tekrar çalıştır.

Örnekler:

```python
hub = PaperHub(provider="openai")     # .env veya built-in OpenAI default model
hub = PaperHub(provider="anthropic")  # .env veya built-in Anthropic default model
hub = PaperHub(provider="google")     # .env veya built-in Google default model

# Modeli özellikle override etmek istersen:
hub = PaperHub(provider="openai", model="gpt-5.4-mini")
```

Gerekli environment variable:

- OpenAI: `OPENAI_API_KEY`
- Anthropic: `ANTHROPIC_API_KEY`
- Google: `GOOGLE_API_KEY`

Daha alt seviye kullanım istersen `build_llm(model=..., provider=...)` ile sadece LLM client oluşturabilirsin; normal kullanımda buna gerek yoktur.

Provider-specific `.env` model değişkenleri:

- `PAPERHUB_ANTHROPIC_MODEL=claude-haiku-4-5-20251001`
- `PAPERHUB_OPENAI_MODEL=gpt-5.4-mini`
- `PAPERHUB_OPENAI_REASONING_EFFORT=xhigh`
- `PAPERHUB_GOOGLE_MODEL=gemini-3-flash-preview`

`PAPERHUB_MODEL` global override olarak hâlâ kullanılabilir. Ama seçili provider ile açıkça uyumsuz bir model id görürse PaperHub onu kullanmaz ve provider default'una döner.


In [ ]:
from paperhub import build_llm

# Bu sadece client nesnesi oluşturur; provider'a istek atmaz.
llm_client = build_llm(model=MODEL, provider=PROVIDER)
print(type(llm_client).__name__, llm_client.provider, llm_client.model)

## 15. En Sık Hatalar

- `Missing API key`: 3. adımdaki API anahtarı hücresini çalıştır veya `.env` dosyanı doldur.
- `model not found`: `MODEL` değerini sağlayıcı panelindeki geçerli model id ile değiştir.
- `No papers found`: Tarihte HuggingFace Daily Papers verisi olmayabilir; `LIVE_RUN_KWARGS` veya `CANDIDATE_RUNS` tarihlerini değiştir.
- PDF metni kısa gelirse paket abstract ile devam etmeye çalışır.
- Bir paper hata verirse tüm çalışma durmaz; ilgili `PaperSummary.error` alanı dolar.

Başlangıç için en güvenli gerçek çalışma:

```python
hub = PaperHub(provider=PROVIDER, model=MODEL, language=LANGUAGE, concurrency=1)
summaries = hub.run(period="day", year=2024, month=5, day=1, top_n=1, display=True)
```
